In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/dias-benchmarks/tpch/notebooks/q17/q17_rewrite/checkpoints/pre_cell_2.pickle

In [4]:
%%RecordEvent
%%time
### cell 2 ###

# Optimized Cell 2
# 1. Filter PART on GPU and select only P_PARTKEY
filtered_part = part[(part.P_BRAND == "Brand#23") & (part.P_CONTAINER == "MED BOX")][["P_PARTKEY"]]

# 2. Compute average L_QUANTITY per L_PARTKEY on GPU and scale by 0.2
lineitem_avg = (
    lineitem[["L_PARTKEY", "L_QUANTITY"]]
    .groupby("L_PARTKEY", as_index=False)
    .mean()
)
lineitem_avg["avg"] = lineitem_avg["L_QUANTITY"] * 0.2
lineitem_avg = lineitem_avg[["L_PARTKEY", "avg"]]

# 3. Merge LINEITEM with filtered PART and with the scaled average
merged = (
    lineitem[["L_PARTKEY", "L_QUANTITY", "L_EXTENDEDPRICE"]]
    .merge(filtered_part, left_on="L_PARTKEY", right_on="P_PARTKEY")
    .merge(lineitem_avg, on="L_PARTKEY")
)

# 4. Filter on the GPU and compute the final metric
filtered = merged[merged.L_QUANTITY < merged.avg]
total = pd.DataFrame({
    "avg_yearly": [filtered.L_EXTENDEDPRICE.sum() / 7.0]
})

CPU times: user 79.6 ms, sys: 35.9 ms, total: 115 ms
Wall time: 175 ms


In [ ]:
%Checkpoint /home/dias-benchmarks/tpch/notebooks/q17/rewritten/o4_mini_high_small/checkpoints/post_cell_2_try_0.pickle